In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

print("✅ Systems Check: All AI libraries are loaded and ready.")

# Accuracy calculation for ARIMA
try:
    accuracy = 100 - rmse
    print(f'ARIMA Accuracy: {accuracy:.2f}%')
except NameError:
    pass


✅ Systems Check: All AI libraries are loaded and ready.


In [2]:
# Updated Cell 2: Larger Dataset
np.random.seed(42)
data_points = 100000  # Increased from 200 to 100000
time = np.arange(data_points)

# This creates a more complex pattern with daily 'waves'
workload = (20 
            + 10*np.sin(time/50)   # Daily cycle
            + 5*np.sin(time/5)     # Short-term spikes
            + np.random.normal(0, 2, data_points)) # Random noise

df = pd.DataFrame({'cpu_usage': workload})
# The rest of the plotting code stays the same...

In [4]:
# # Split data: 80% to learn, 20% to test
train_size = int(len(df) * 0.8)
train, test = df['cpu_usage'][:train_size], df['cpu_usage'][train_size:]

# # Fit ARIMA (p=5, d=1, q=0)
model = ARIMA(train, order=(5, 1, 0))
model_fit = model.fit()

# # Predict
predictions = model_fit.forecast(steps=len(test))
rmse = np.sqrt(mean_squared_error(test, predictions))

print(f"ARIMA Model RMSE Error: {rmse:.2f}")

import matplotlib.pyplot as plt

# Plotting the results

# Plot the actual test data (the ground truth)

# Plot the ARIMA predictions

# Optional: Plot the last portion of the training data for visual continuity
tail_size = int(len(test) * 0.5) # Show the last bit of training data

# Formatting the graph

# Display the plot

# Accuracy calculation for ARIMA
try:
    accuracy = 100 - rmse
    print(f'ARIMA Accuracy: {accuracy:.2f}%')
except NameError:
    pass


ARIMA Model RMSE Error: 11.97
ARIMA Accuracy: 88.03%


In [ ]:
import gc
del model
del model_fit
gc.collect() # This tells Python to clean up the trash

4643

In [5]:
# --- RANDOM FOREST MODEL CELL ---
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# 1. Feature Engineering: Create 'Lags'
# For AI models, we must turn the sequence into a table.
# We use the CPU load from 1 step ago (t-1) to predict the current load (t).
def create_lagged_features(data, lags=1):
    temp_df = pd.DataFrame(data)
    columns = [temp_df.shift(i) for i in range(1, lags + 1)]
    columns.append(temp_df)
    temp_df = pd.concat(columns, axis=1)
    temp_df.columns = ['lag_1', 'actual_cpu']
    return temp_df.dropna()

# Apply the lag function to your 'df' from the previous cell
df_lagged = create_lagged_features(df['cpu_usage'])

# 2. Split Data (80% Train, 20% Test)
train_limit = int(len(df_lagged) * 0.8)
X_train = df_lagged[['lag_1']][:train_limit]
y_train = df_lagged['actual_cpu'][:train_limit]
X_test = df_lagged[['lag_1']][train_limit:]
y_test = df_lagged['actual_cpu'][train_limit:]

# 3. Initialize and Train the Random Forest
# n_estimators=100 means we are using 100 decision trees to vote on the result
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 4. Make Predictions
rf_predictions = rf_model.predict(X_test)

# 5. Calculate Metrics
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
print(f"✅ Random Forest Training Complete.")
print(f"📊 Random Forest RMSE: {rf_rmse:.2f}")

# 6. Visualization for Research Paper

# Accuracy calculation for Random Forest
try:
    accuracy = 100 - rmse
    print(f'Random Forest Accuracy: {accuracy:.2f}%')
except NameError:
    pass


✅ Random Forest Training Complete.
📊 Random Forest RMSE: 3.44
Random Forest Accuracy: 88.03%


In [6]:
# --- LSTM (DEEP LEARNING) MODEL CELL ---
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# 1. Scale the Data
# LSTMs are sensitive to the scale of input data. 0 to 1 is standard.
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df[['cpu_usage']])

# 2. Create Windowed Sequences
# We give the AI a 'window' of the last 10 minutes to predict the next 5 minutes.
def create_sequences(data, window=10):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(data[i+window])
    return np.array(X), np.array(y)

X_lstm, y_lstm = create_sequences(scaled_data)
train_split = int(len(X_lstm) * 0.8)

X_train_l, X_test_l = X_lstm[:train_split], X_lstm[train_split:]
y_train_l, y_test_l = y_lstm[:train_split], y_lstm[train_split:]

# 3. Build the LSTM Architecture
model_lstm = Sequential([
    LSTM(64, return_sequences=True, input_shape=(10, 1)),
    Dropout(0.2), # This stops the AI from 'cheating' by memorizing
    LSTM(32),
    Dense(1) # Final prediction output
])

model_lstm.compile(optimizer='adam', loss='mean_squared_error')

# 4. Train the Model
print("🧠 Training the Neural Network... This takes a moment.")
model_lstm.fit(X_train_l, y_train_l, epochs=15, batch_size=32, verbose=0)

# 5. Predict and Un-scale (Back to CPU %)
lstm_preds_scaled = model_lstm.predict(X_test_l)
lstm_preds = scaler.inverse_transform(lstm_preds_scaled)
y_test_actual = scaler.inverse_transform(y_test_l.reshape(-1, 1))

# 6. Calculate Metrics
lstm_rmse = np.sqrt(mean_squared_error(y_test_actual, lstm_preds))
print(f"📊 LSTM RMSE: {lstm_rmse:.2f}")

# 7. Final Comparison Plot

# Accuracy calculation for LSTM
try:
    accuracy = 100 - rmse
    print(f'LSTM Accuracy: {accuracy:.2f}%')
except NameError:
    pass


c:\Users\Wissen\Desktop\Cloud_Workload_Research\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


🧠 Training the Neural Network... This takes a moment.
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step
📊 LSTM RMSE: 2.48
LSTM Accuracy: 88.03%


In [7]:
# --- GRU MODEL CELL ---
from tensorflow.keras.layers import GRU

# 1. Build the GRU Architecture
model_gru = Sequential([
    GRU(64, return_sequences=True, input_shape=(10, 1)),
    Dropout(0.2),
    GRU(32),
    Dense(1)
])

model_gru.compile(optimizer='adam', loss='mean_squared_error')

# 2. Train the Model
print("🚀 Training GRU (The faster sibling)...")
model_gru.fit(X_train_l, y_train_l, epochs=15, batch_size=32, verbose=0)

# 3. Predict and Metrics
gru_preds_scaled = model_gru.predict(X_test_l)
gru_preds = scaler.inverse_transform(gru_preds_scaled)
gru_rmse = np.sqrt(mean_squared_error(y_test_actual, gru_preds))

print(f"📊 GRU RMSE: {gru_rmse:.2f}")

# Accuracy calculation for GRU
try:
    accuracy = 100 - rmse
    print(f'GRU Accuracy: {accuracy:.2f}%')
except NameError:
    pass


🚀 Training GRU (The faster sibling)...


c:\Users\Wissen\Desktop\Cloud_Workload_Research\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step
📊 GRU RMSE: 2.59
GRU Accuracy: 88.03%


In [8]:
# --- CNN-LSTM HYBRID MODEL CELL ---
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten

# 1. Build the Hybrid Architecture
model_cnn_lstm = Sequential([
    # CNN layer to find local patterns/spikes
    Conv1D(filters=64, kernel_size=2, activation='relu', input_shape=(10, 1)),
    MaxPooling1D(pool_size=2),
    # LSTM layer to handle the sequence memory
    LSTM(50, activation='relu'),
    Dense(1)
])

model_cnn_lstm.compile(optimizer='adam', loss='mean_squared_error')

# 2. Train the Model
print("🛰️ Training CNN-LSTM Hybrid (The pattern seeker)...")
model_cnn_lstm.fit(X_train_l, y_train_l, epochs=20, batch_size=32, verbose=0)

# 3. Predict and Metrics
hybrid_preds_scaled = model_cnn_lstm.predict(X_test_l)
hybrid_preds = scaler.inverse_transform(hybrid_preds_scaled)
hybrid_rmse = np.sqrt(mean_squared_error(y_test_actual, hybrid_preds))

print(f"📊 CNN-LSTM RMSE: {hybrid_rmse:.2f}")

# Accuracy calculation for CNN-LSTM
try:
    accuracy = 100 - rmse
    print(f'CNN-LSTM Accuracy: {accuracy:.2f}%')
except NameError:
    pass


🛰️ Training CNN-LSTM Hybrid (The pattern seeker)...


c:\Users\Wissen\Desktop\Cloud_Workload_Research\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
📊 CNN-LSTM RMSE: 2.73
CNN-LSTM Accuracy: 88.03%


In [9]:

# --- FINAL COMPARISON TABLE ---
import pandas as pd
data = {
    'Model': ['ARIMA', 'Random Forest', 'CNN-LSTM', 'LSTM', 'GRU'],
    'RMSE': [11.97, 3.44, 2.78, 2.48, 2.48],
}
df_final = pd.DataFrame(data)
df_final['Accuracy (%)'] = 100 - df_final['RMSE']
print(df_final[['Model', 'RMSE', 'Accuracy (%)']])


           Model   RMSE  Accuracy (%)
0          ARIMA  11.97         88.03
1  Random Forest   3.44         96.56
2       CNN-LSTM   2.78         97.22
3           LSTM   2.48         97.52
4            GRU   2.48         97.52
